In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Exploratory Data Analysis — Brugada-HUCA Dataset\n",
    "**IDSC 2026 | Team MediScript**\n",
    "\n",
    "This notebook explores the Brugada-HUCA dataset before modeling:\n",
    "- Class distribution and imbalance\n",
    "- Signal quality verification\n",
    "- Sample ECG visualization per class\n",
    "- Lead-wise amplitude statistics"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys, os\n",
    "sys.path.insert(0, os.path.dirname(os.getcwd()))\n",
    "\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import matplotlib.gridspec as gridspec\n",
    "import seaborn as sns\n",
    "import wfdb\n",
    "\n",
    "from config import DATA_DIR, METADATA_PATH\n",
    "from src.preprocessing import merge_labels\n",
    "\n",
    "plt.rcParams['figure.dpi'] = 120\n",
    "print('Libraries loaded successfully.')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Load Metadata"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "df = pd.read_csv(METADATA_PATH)\n",
    "print(f'Total records: {len(df)}')\n",
    "print(f'\\nLabel distribution (raw):')\n",
    "print(df['brugada'].value_counts())\n",
    "print(f'\\nMetadata columns: {list(df.columns)}')\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. Class Distribution"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Apply label merge (label 2 → 1)\n",
    "df['label_merged'] = merge_labels(df['brugada'].tolist())\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "\n",
    "# Raw labels\n",
    "raw_counts = df['brugada'].value_counts().sort_index()\n",
    "axes[0].bar(['Normal (0)', 'Brugada (1)', 'Borderline (2)'],\n",
    "            [raw_counts.get(0,0), raw_counts.get(1,0), raw_counts.get(2,0)],\n",
    "            color=['steelblue', 'crimson', 'orange'], edgecolor='white')\n",
    "axes[0].set_title('Raw Label Distribution', fontweight='bold')\n",
    "axes[0].set_ylabel('Count')\n",
    "for i, v in enumerate([raw_counts.get(0,0), raw_counts.get(1,0), raw_counts.get(2,0)]):\n",
    "    axes[0].text(i, v + 2, str(v), ha='center', fontweight='bold')\n",
    "\n",
    "# Merged labels\n",
    "merged_counts = df['label_merged'].value_counts().sort_index()\n",
    "axes[1].bar(['Normal (0)', 'Brugada (1)'],\n",
    "            [merged_counts.get(0,0), merged_counts.get(1,0)],\n",
    "            color=['steelblue', 'crimson'], edgecolor='white')\n",
    "axes[1].set_title('After Merging Label 2 → 1', fontweight='bold')\n",
    "axes[1].set_ylabel('Count')\n",
    "for i, v in enumerate([merged_counts.get(0,0), merged_counts.get(1,0)]):\n",
    "    axes[1].text(i, v + 2, str(v), ha='center', fontweight='bold')\n",
    "axes[1].text(0.5, 0.85, f'Imbalance ratio: {merged_counts.get(0,0)/merged_counts.get(1,0):.2f}:1',\n",
    "             transform=axes[1].transAxes, ha='center', color='gray')\n",
    "\n",
    "plt.suptitle('Brugada-HUCA Class Distribution', fontsize=13, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../outputs/figures/eda_class_distribution.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "print(f'Imbalance ratio: {merged_counts.get(0,0)/merged_counts.get(1,0):.2f}:1')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Signal Quality Check"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('Checking all records...')\n",
    "shapes, nan_count, failed = [], 0, []\n",
    "\n",
    "for pid in df['patient_id'].astype(str):\n",
    "    try:\n",
    "        rec = wfdb.rdrecord(f'{DATA_DIR}/{pid}/{pid}')\n",
    "        shapes.append(rec.p_signal.shape)\n",
    "        if np.isnan(rec.p_signal).any():\n",
    "            nan_count += 1\n",
    "    except Exception as e:\n",
    "        failed.append((pid, str(e)))\n",
    "\n",
    "print(f'Successfully loaded : {len(df) - len(failed)}/{len(df)}')\n",
    "print(f'Failed              : {len(failed)}')\n",
    "print(f'Records with NaN    : {nan_count}')\n",
    "print(f'Unique signal shapes: {set(shapes)}')\n",
    "print(f'Sampling frequency  : {wfdb.rdrecord(f\"{DATA_DIR}/{df.patient_id.astype(str).iloc[0]}/{df.patient_id.astype(str).iloc[0]}\").fs} Hz')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Sample ECG Visualization — One Normal, One Brugada"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "lead_names = ['I','II','III','aVR','aVL','aVF','V1','V2','V3','V4','V5','V6']\n",
    "\n",
    "normal_id  = df[df['label_merged']==0]['patient_id'].iloc[0]\n",
    "brugada_id = df[df['label_merged']==1]['patient_id'].iloc[0]\n",
    "\n",
    "def load_signal(pid):\n",
    "    rec = wfdb.rdrecord(f'{DATA_DIR}/{pid}/{pid}')\n",
    "    return rec.p_signal.T  # (12, 1200)\n",
    "\n",
    "normal_sig  = load_signal(str(normal_id))\n",
    "brugada_sig = load_signal(str(brugada_id))\n",
    "\n",
    "fig, axes = plt.subplots(12, 2, figsize=(16, 20), sharey=False)\n",
    "t = np.arange(1200) / 100  # time in seconds\n",
    "\n",
    "for i, lead in enumerate(lead_names):\n",
    "    axes[i,0].plot(t, normal_sig[i],  color='steelblue', lw=0.8)\n",
    "    axes[i,1].plot(t, brugada_sig[i], color='crimson',   lw=0.8)\n",
    "    axes[i,0].set_ylabel(lead, fontsize=8)\n",
    "    axes[i,0].tick_params(labelsize=7)\n",
    "    axes[i,1].tick_params(labelsize=7)\n",
    "    axes[i,0].grid(alpha=0.3)\n",
    "    axes[i,1].grid(alpha=0.3)\n",
    "\n",
    "axes[0,0].set_title(f'Normal (Patient {normal_id})',  fontweight='bold', color='steelblue')\n",
    "axes[0,1].set_title(f'Brugada (Patient {brugada_id})', fontweight='bold', color='crimson')\n",
    "axes[-1,0].set_xlabel('Time (s)')\n",
    "axes[-1,1].set_xlabel('Time (s)')\n",
    "\n",
    "plt.suptitle('12-Lead ECG: Normal vs. Brugada\\n(Note ST-segment differences in V1–V3)',\n",
    "             fontsize=12, fontweight='bold', y=1.005)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../outputs/figures/eda_ecg_comparison.png', dpi=120, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 5. Lead-Wise Amplitude Distribution by Class"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print('Computing per-lead statistics (this takes ~2 minutes)...')\n",
    "stats = {'lead': [], 'class': [], 'mean': [], 'std': []}\n",
    "\n",
    "for _, row in df.iterrows():\n",
    "    pid = str(row['patient_id'])\n",
    "    label = row['label_merged']\n",
    "    try:\n",
    "        sig = load_signal(pid)\n",
    "        for i, lead in enumerate(lead_names):\n",
    "            stats['lead'].append(lead)\n",
    "            stats['class'].append('Brugada' if label else 'Normal')\n",
    "            stats['mean'].append(np.mean(sig[i]))\n",
    "            stats['std'].append(np.std(sig[i]))\n",
    "    except:\n",
    "        pass\n",
    "\n",
    "stats_df = pd.DataFrame(stats)\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
    "sns.boxplot(data=stats_df, x='lead', y='mean', hue='class',\n",
    "            palette={'Normal':'steelblue','Brugada':'crimson'}, ax=axes[0])\n",
    "axes[0].set_title('Mean Amplitude per Lead by Class', fontweight='bold')\n",
    "axes[0].set_xlabel('Lead'); axes[0].set_ylabel('Mean Amplitude (mV)')\n",
    "axes[0].tick_params(axis='x', rotation=45)\n",
    "\n",
    "sns.boxplot(data=stats_df, x='lead', y='std', hue='class',\n",
    "            palette={'Normal':'steelblue','Brugada':'crimson'}, ax=axes[1])\n",
    "axes[1].set_title('Signal Std Dev per Lead by Class', fontweight='bold')\n",
    "axes[1].set_xlabel('Lead'); axes[1].set_ylabel('Std Dev (mV)')\n",
    "axes[1].tick_params(axis='x', rotation=45)\n",
    "\n",
    "plt.suptitle('Lead-wise Amplitude Statistics: Normal vs. Brugada', fontsize=12, fontweight='bold')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../outputs/figures/eda_lead_statistics.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "print('EDA complete.')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.12.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}